# Pharmacokinetics & Pharmacodynamics (PK/PD) Modeling
## A Computational Introduction for Biomedical Scientists

**Course:** UConn Systems Biology — Fall 2026  
**Duration:** 1–2 hours  
**Prerequisites:** Basic Python, Jupyter notebooks

---

## Learning Objectives

By the end of this lecture, you will be able to:

1. **Explain** the conceptual difference between pharmacokinetics (PK) and pharmacodynamics (PD)
2. **Simulate** drug concentration–time profiles using analytic solutions to PK models
3. **Fit** a one-compartment oral PK model to real data using `scipy.optimize.curve_fit`
4. **Interpret** fitted parameters (clearance, volume of distribution, absorption rate)
5. **Quantify** inter-subject variability in PK parameters across a population
6. **Simulate** a pharmacodynamic Emax model and link it to a PK profile
7. **Numerically solve** ODEs with `scipy.integrate.solve_ivp` and recognise when analytic solutions break down

---

> **A note on tools:** This notebook uses pure Python — `numpy`, `pandas`, `matplotlib`, and `scipy`. No R, no NONMEM, no special PK software required.

## Setup

Run this cell first. It imports every library we will need and sets some sensible plotting defaults.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp
from io import StringIO

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('All imports successful.')

ModuleNotFoundError: No module named 'numpy'

---
# Part 1: The Big Picture — PK vs PD

## What are we modeling, and why?

Every time a drug is administered, two broad processes happen in sequence:

| Process | Full name | Question it answers | Governed by |
|---------|-----------|---------------------|-------------|
| **PK** | Pharmacokinetics | What does the **body** do to the **drug**? | ADME: Absorption, Distribution, Metabolism, Excretion |
| **PD** | Pharmacodynamics | What does the **drug** do to the **body**? | Drug–receptor binding, downstream signalling |

Put simply:

$$\text{Dose} \xrightarrow{\text{PK}} \text{Concentration}(t) \xrightarrow{\text{PD}} \text{Effect}(t)$$

A PK/PD model links all three stages mathematically.

### Conceptual diagram

The figure below shows the full dose–concentration–effect chain. Keep this mental model in mind throughout the lecture.

![PK/PD diagram](images/pkpd-diagram.png)

*(If the image does not render, the file `images/pkpd-diagram.png` may be missing from your working directory — that is fine, the text description above is the key idea.)*

### What PK and PD curves look like in practice

![PK/PD concept plots](images/pkpd-plots-concept.png)

The **left panel** shows a typical plasma concentration–time curve after an oral dose: the drug is absorbed (rising phase), reaches a peak (Cmax), then is eliminated (falling phase). The **right panel** shows the corresponding effect: it rises with concentration but saturates because receptors become fully occupied.

## Why build a model instead of just looking at the data?

| Approach | What you get | What you miss |
|----------|-------------|---------------|
| **Raw data only** | What happened in *this* experiment | Cannot predict a different dose, different schedule, or a different patient |
| **Mechanistic model** | Parameter estimates (CL, V, Emax, EC50) | Requires assumptions about model structure |

A fitted model lets you **simulate** scenarios that were never actually run — for example, "what concentration would we see if we gave a 10 mg/kg dose every 8 hours?" or "at what dose does 90% of maximal effect occur?"

This is the core power of PK/PD modeling in drug development and precision medicine.

---
# Part 2: PK Models

## The one-compartment model

The simplest useful model treats the body as a **single well-mixed compartment**. Think of it as a tank of water: drug flows in, mixes instantaneously, and drains at a rate proportional to how much is present.

For an **IV bolus** (drug injected directly into the bloodstream, so there is no absorption phase):

$$\frac{dC}{dt} = -k_e \cdot C, \qquad C(0) = \frac{\text{Dose}}{V}$$

This ODE has the well-known analytic solution:

$$C(t) = \frac{\text{Dose}}{V} \cdot e^{-k_e t}$$

### Parameter glossary

| Symbol | Name | Units | Interpretation |
|--------|------|-------|----------------|
| $V$ | Volume of distribution | L or L/kg | How widely the drug distributes; **not** a real anatomical volume |
| $CL$ | Clearance | L/h or L/h/kg | Rate at which drug is permanently removed from the body |
| $k_e$ | Elimination rate constant | 1/h | $k_e = CL / V$ |
| $t_{1/2}$ | Half-life | h | Time for concentration to halve; $t_{1/2} = \ln(2) / k_e$ |

> **Key insight:** $CL$ and $V$ are the *primary* parameters — they have physiological meaning. $k_e$ and $t_{1/2}$ are derived from them.

### Simulating the IV bolus model

Let us simulate the analytic solution and explore how changing each parameter shifts the concentration–time curve. We will:

1. Fix $V = 20$ L and vary $CL$ — this changes how fast the drug is cleared
2. Fix $CL = 2$ L/h and vary $V$ — this changes how diluted the drug is

In [ ]:
# --- 1-compartment IV bolus: analytic solution ---

def pk_iv_1cpt(t, dose, V, CL):
    ke = CL / V
    return (dose / V) * np.exp(-ke * t)

t = np.linspace(0, 24, 300)   # time vector: 0 to 24 hours
dose = 100                      # mg

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Panel A: vary CL, fix V ---
V_fixed = 20
for CL_val in [1, 2, 4]:
    ke = CL_val / V_fixed
    t_half = np.log(2) / ke
    C = pk_iv_1cpt(t, dose, V_fixed, CL_val)
    axes[0].plot(t, C, label=f'CL={CL_val} L/h  (t½={t_half:.1f} h)')

axes[0].set_xlabel('Time (h)')
axes[0].set_ylabel('Concentration (mg/L)')
axes[0].set_title('Effect of varying CL  (V = 20 L fixed)')
axes[0].legend()
axes[0].set_ylim(bottom=0)

# --- Panel B: vary V, fix CL ---
CL_fixed = 2
for V_val in [10, 20, 40]:
    ke = CL_fixed / V_val
    t_half = np.log(2) / ke
    C = pk_iv_1cpt(t, dose, V_val, CL_fixed)
    axes[1].plot(t, C, label=f'V={V_val} L  (t½={t_half:.1f} h)')

axes[1].set_xlabel('Time (h)')
axes[1].set_ylabel('Concentration (mg/L)')
axes[1].set_title('Effect of varying V  (CL = 2 L/h fixed)')
axes[1].legend()
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.suptitle('1-Compartment IV Bolus Model', y=1.02, fontsize=13, fontweight='bold')
plt.show()

### Interpretation

**Left panel (varying CL):** Higher clearance means faster elimination — the curve drops more steeply and the half-life is shorter. Notice that the initial concentration $C_0 = \text{Dose}/V$ is **unchanged** because $V$ is fixed.

**Right panel (varying V):** A larger volume of distribution means the drug is more widely distributed, so the initial plasma concentration is **lower** (same dose, more dilution). However, because $CL$ is fixed, the total amount cleared per unit time is the same — the half-life is actually *longer* because there is more drug to clear but the same clearing capacity.

## One-compartment oral absorption model

For an **oral dose**, the drug must first be absorbed from the gut into the bloodstream. We model the gut as a separate depot compartment:

$$\frac{dD_{\text{gut}}}{dt} = -k_a \cdot D_{\text{gut}}, \qquad D_{\text{gut}}(0) = F \cdot \text{Dose}$$

$$\frac{dC}{dt} = \frac{k_a \cdot D_{\text{gut}}}{V} - k_e \cdot C, \qquad C(0) = 0$$

Here $k_a$ is the first-order absorption rate constant and $F$ is bioavailability (fraction absorbed). The analytic solution is:

$$C(t) = \frac{F \cdot \text{Dose} \cdot k_a}{V(k_a - k_e)} \left( e^{-k_e t} - e^{-k_a t} \right)$$

> **Important:** This formula requires $k_a \neq k_e$. When $k_a \gg k_e$, absorption is fast and the terminal slope reflects elimination. When $k_a \approx k_e$ ("flip-flop" kinetics), you cannot tell absorption from elimination by the slope alone — you need additional information (e.g., an IV study).

### Simulating the oral model — effect of varying $k_a$

Let us fix $CL$, $V$, and $F$, and change only $k_a$. Watch how the shape of the rise-and-fall changes.

In [ ]:
# --- 1-compartment oral: analytic solution ---

def pk_oral_1cpt_analytic(t, dose, F, V, CL, ka):
    ke = CL / V
    if np.isclose(ka, ke):
        # Limiting case: ka == ke (use L'Hopital)
        return (F * dose * ka / V) * t * np.exp(-ke * t)
    return (F * dose * ka) / (V * (ka - ke)) * (np.exp(-ke * t) - np.exp(-ka * t))

t = np.linspace(0, 24, 400)
dose = 4.0    # mg/kg  (mimicking the Theophylline dataset we will fit later)
F    = 1.0    # assume complete bioavailability for now
V    = 0.5    # L/kg
CL   = 0.04   # L/h/kg

fig, ax = plt.subplots(figsize=(8, 4))

for ka in [0.5, 1.5, 4.0, 10.0]:
    C = pk_oral_1cpt_analytic(t, dose, F, V, CL, ka)
    ax.plot(t, C, label=f'ka = {ka} h⁻¹')

# Mark the ke for reference
ke = CL / V
ax.axvline(np.log(2) / ke, color='grey', linestyle=':', alpha=0.7, label=f'ke = {ke:.3f} h⁻¹  (t½={np.log(2)/ke:.1f} h)')

ax.set_xlabel('Time (h)')
ax.set_ylabel('Concentration (mg/L per mg/kg dose... or mg/kg·L/kg = mg/L)')
ax.set_title('1-Compartment Oral Model — Effect of Varying ka')
ax.legend(fontsize=9)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

### Interpretation

- **Small $k_a$ (slow absorption, e.g., 0.5 h⁻¹):** The peak is lower and delayed, and the rising phase is gradual. When $k_a < k_e$, the terminal slope actually reflects *absorption*, not elimination — this is called **flip-flop kinetics**.
- **Large $k_a$ (fast absorption, e.g., 10 h⁻¹):** Absorption is nearly instantaneous; the curve quickly reaches a peak and then declines almost like an IV bolus. The terminal slope is determined by $k_e$.
- **Moderate $k_a$ (1.5 h⁻¹):** A characteristic bell-shaped rise-and-fall typical of oral dosing.

The dashed vertical line marks the half-life based on $k_e$ — notice how for slow absorption, the peak is reached after the half-life.

---
# Part 3: Fitting Real PK Data

## The Theophylline dataset

Theophylline is a bronchodilator (used in asthma treatment) with a relatively narrow therapeutic window — clinically important to model. This is a classic PK dataset containing plasma concentration measurements from **6 subjects** who each received a single oral dose.

We will:
1. Load and visualise all subjects
2. Focus on Subject 1
3. Fit the oral PK model to Subject 1 using nonlinear least squares
4. Interpret the fitted parameters

### Loading the data

The dataset is embedded directly in the notebook as a CSV string so there is no external file dependency.

In [ ]:
theoph_csv = '''
Subject,Wt,Dose,Time,conc
1,79.6,4.02,0.00,0.74
1,79.6,4.02,0.25,2.84
1,79.6,4.02,0.57,6.57
1,79.6,4.02,1.12,10.50
1,79.6,4.02,2.02,9.66
1,79.6,4.02,3.82,8.58
1,79.6,4.02,5.10,8.36
1,79.6,4.02,7.03,7.47
1,79.6,4.02,9.05,6.89
1,79.6,4.02,12.12,5.94
1,79.6,4.02,24.37,3.28
2,72.4,4.40,0.00,0.00
2,72.4,4.40,0.27,1.72
2,72.4,4.40,0.52,7.91
2,72.4,4.40,1.00,8.31
2,72.4,4.40,1.92,8.33
2,72.4,4.40,3.50,6.85
2,72.4,4.40,5.02,6.08
2,72.4,4.40,7.03,5.40
2,72.4,4.40,9.00,4.55
2,72.4,4.40,12.00,3.01
2,72.4,4.40,24.30,0.90
3,70.5,4.53,0.00,0.00
3,70.5,4.53,0.27,4.40
3,70.5,4.53,0.58,6.90
3,70.5,4.53,1.02,8.20
3,70.5,4.53,2.02,7.80
3,70.5,4.53,3.62,7.00
3,70.5,4.53,5.08,6.20
3,70.5,4.53,7.07,5.30
3,70.5,4.53,9.00,4.90
3,70.5,4.53,12.15,3.70
3,70.5,4.53,24.17,1.05
4,72.7,4.40,0.00,0.00
4,72.7,4.40,0.35,1.89
4,72.7,4.40,0.60,4.60
4,72.7,4.40,1.07,8.60
4,72.7,4.40,2.13,8.38
4,72.7,4.40,3.50,7.54
4,72.7,4.40,5.02,6.88
4,72.7,4.40,7.02,5.78
4,72.7,4.40,9.02,5.33
4,72.7,4.40,11.98,4.19
4,72.7,4.40,24.65,1.15
5,54.6,5.86,0.00,0.00
5,54.6,5.86,0.30,2.02
5,54.6,5.86,0.52,5.63
5,54.6,5.86,1.00,11.40
5,54.6,5.86,2.02,9.33
5,54.6,5.86,3.50,8.74
5,54.6,5.86,5.02,7.56
5,54.6,5.86,7.02,7.09
5,54.6,5.86,9.10,5.90
5,54.6,5.86,12.00,4.37
5,54.6,5.86,24.35,1.57
6,80.0,4.00,0.00,0.00
6,80.0,4.00,0.27,1.29
6,80.0,4.00,0.58,3.08
6,80.0,4.00,1.02,6.44
6,80.0,4.00,2.02,6.32
6,80.0,4.00,3.53,5.53
6,80.0,4.00,5.05,4.94
6,80.0,4.00,7.15,4.02
6,80.0,4.00,9.22,3.46
6,80.0,4.00,12.10,2.78
6,80.0,4.00,23.85,0.92
'''

theoph = pd.read_csv(StringIO(theoph_csv))
theoph['Subject'] = theoph['Subject'].astype(int)
print(theoph.head(12))
print(f'\nTotal rows: {len(theoph)}  |  Subjects: {theoph["Subject"].nunique()}')

### Spaghetti plot: all 6 subjects

Before fitting anything, let us look at all subjects together. This is called a **spaghetti plot** in pharmacometrics — each line is one subject. It immediately reveals inter-subject variability.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

colors = plt.cm.tab10.colors

for i, subj_id in enumerate(sorted(theoph['Subject'].unique())):
    subj = theoph[theoph['Subject'] == subj_id]
    dose_val = subj['Dose'].iloc[0]
    ax.plot(subj['Time'], subj['conc'], 'o-',
            color=colors[i], label=f'Subject {subj_id}  (dose={dose_val} mg/kg)',
            linewidth=1.5, markersize=5)

ax.set_xlabel('Time (h)')
ax.set_ylabel('Theophylline concentration (mg/L)')
ax.set_title('Theophylline PK — All Subjects')
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

### Interpretation

All 6 subjects received similar oral doses (4–6 mg/kg) yet show noticeably different concentration–time profiles:

- **Peak concentration (Cmax)** ranges from about 8 to 11 mg/L
- **Time to peak (Tmax)** ranges from about 1 to 2 h
- **Terminal slope** varies — some subjects eliminate faster than others

This is **inter-subject variability** — driven by differences in body size, renal function, liver enzyme activity, genetics, and other factors. Part 4 will quantify this formally.

### Focus: Subject 1

Subject 1 is interesting because the **first time point (t = 0) has a non-zero concentration of 0.74 mg/L** — this suggests a small residual amount from a previous dose or a pre-dose sample timing artefact. We will include it in the fit to see how the model handles it.

Let us plot Subject 1 in isolation to appreciate the rise-and-fall shape that the oral model needs to capture.

In [ ]:
subj1 = theoph[theoph['Subject'] == 1].copy()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(subj1['Time'], subj1['conc'], 'ko-', markersize=7, label='Observed')
ax.set_xlabel('Time (h)')
ax.set_ylabel('Concentration (mg/L)')
ax.set_title('Subject 1 — Theophylline Concentration vs Time')
ax.legend()
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

print(f'Dose for Subject 1: {subj1["Dose"].iloc[0]} mg/kg')

### Fitting the 1-compartment oral model to Subject 1

We will use `scipy.optimize.curve_fit`, which performs **nonlinear least squares** optimisation. It finds the parameter values that minimise the sum of squared residuals between the model predictions and the observed data.

The function signature must be `f(t, param1, param2, ...)` — `curve_fit` will optimise the parameters. Because the dose is fixed (not a free parameter), we capture it in a closure.

Initial guesses: $k_a = 1.5$ h⁻¹, $CL = 0.04$ L/h/kg, $V = 0.5$ L/kg — reasonable for theophylline based on literature.

In [ ]:
# Extract Subject 1 data
t_obs  = subj1['Time'].values
C_obs  = subj1['conc'].values
dose_s1 = subj1['Dose'].iloc[0]   # mg/kg
F = 1.0  # assume complete bioavailability

# Define the model function in the form curve_fit expects: f(t, ka, CL, V)
def pk_oral_1cpt(t, ka, CL, V):
    ke = CL / V
    if np.isclose(ka, ke, rtol=1e-6):
        return (F * dose_s1 * ka / V) * t * np.exp(-ke * t)
    return (F * dose_s1 * ka) / (V * (ka - ke)) * (np.exp(-ke * t) - np.exp(-ka * t))

# Vectorise over t (curve_fit passes a numpy array)
pk_oral_1cpt_vec = np.vectorize(pk_oral_1cpt, excluded=['ka', 'CL', 'V'])

def pk_oral_1cpt_fit(t, ka, CL, V):
    return pk_oral_1cpt_vec(t, ka=ka, CL=CL, V=V)

# Initial guesses and bounds
p0     = [1.5, 0.04, 0.5]
bounds = ([0.01, 0.001, 0.05], [20.0, 1.0, 5.0])

popt, pcov = curve_fit(pk_oral_1cpt_fit, t_obs, C_obs, p0=p0, bounds=bounds, maxfev=10000)

ka_fit, CL_fit, V_fit = popt
ke_fit = CL_fit / V_fit
t_half_fit = np.log(2) / ke_fit

print('=== Fitted PK parameters for Subject 1 ===')
print(f'  ka  = {ka_fit:.3f} h⁻¹   (absorption rate constant)')
print(f'  CL  = {CL_fit:.4f} L/h/kg   (clearance per kg body weight)')
print(f'  V   = {V_fit:.3f} L/kg   (volume of distribution per kg)')
print(f'  ke  = CL/V = {ke_fit:.4f} h⁻¹   (derived elimination rate)')
print(f'  t½  = ln(2)/ke = {t_half_fit:.2f} h   (half-life)')

# Total CL and V (multiply by body weight)
bw = subj1['Wt'].iloc[0]
print(f'\n  Body weight: {bw} kg')
print(f'  Total CL = {CL_fit * bw:.3f} L/h')
print(f'  Total V  = {V_fit  * bw:.2f} L')

### Plotting observed vs predicted

Now let us overlay the model prediction on the observed data to assess the goodness of fit visually.

In [ ]:
t_pred = np.linspace(0, 25, 400)
C_pred = pk_oral_1cpt_fit(t_pred, *popt)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(t_obs, C_obs, color='black', zorder=5, s=50, label='Observed')
ax.plot(t_pred, C_pred, 'r-', linewidth=2, label='Model fit')

# Annotate key parameters on the plot
ax.text(14, 9.0, f'ka = {ka_fit:.2f} h⁻¹', fontsize=9)
ax.text(14, 8.0, f'CL = {CL_fit:.4f} L/h/kg', fontsize=9)
ax.text(14, 7.0, f'V  = {V_fit:.3f} L/kg', fontsize=9)
ax.text(14, 6.0, f't½ = {t_half_fit:.1f} h', fontsize=9)

ax.set_xlabel('Time (h)')
ax.set_ylabel('Concentration (mg/L)')
ax.set_title('Subject 1 — Observed vs Fitted (1-cpt Oral Model)')
ax.legend()
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

### Interpretation

The fitted model captures the general shape of Subject 1's profile well:

- The **absorption phase** (rapid rise to ~10 mg/L) is described by $k_a$
- The **elimination phase** (slow decline from peak) is described by $k_e = CL/V$
- The **half-life of ~8–9 hours** is consistent with known theophylline pharmacokinetics (literature: 6–10 h in healthy adults)

Note that the t = 0 point (0.74 mg/L, likely a pre-dose residual) is slightly above the model prediction of 0 — the model assumes the compartment starts empty, so this small discrepancy is expected and acceptable.

**Clinical context:** Theophylline has a therapeutic range of approximately 5–15 mg/L. Looking at our prediction, the concentration stays within this range from roughly 1 to >24 hours after dosing — useful information for dosing interval design.

---
# Part 4: Population PK

## Inter-subject variability

In Part 3, we fit the model to a single subject. But in real pharmacology, we care about the **population**: how do parameters vary from person to person, and what is the "typical" patient?

The gold-standard approach is **nonlinear mixed-effects modeling (NLME)**, implemented in tools like NONMEM or the R package `nlmixr2`. NLME estimates:
- **Fixed effects:** the population-mean parameters
- **Random effects:** the distribution of deviations around the mean for each individual

Here we will approximate this idea by fitting each subject independently and then examining the distribution of the fitted parameters. This is called the **two-stage approach** — it overestimates variability slightly, but it conveys the core concept.

### Fitting all 6 subjects in a loop

We reuse the same `pk_oral_1cpt_fit` structure but re-define the closure inside the loop for each subject's dose.

In [ ]:
results = []
p0     = [1.5, 0.04, 0.5]
bounds = ([0.01, 0.001, 0.05], [20.0, 1.0, 5.0])
F = 1.0

for subj_id in sorted(theoph['Subject'].unique()):
    subj_data = theoph[theoph['Subject'] == subj_id]
    t_s = subj_data['Time'].values
    C_s = subj_data['conc'].values
    dose_s = subj_data['Dose'].iloc[0]
    bw_s   = subj_data['Wt'].iloc[0]

    # Build model function with this subject's dose captured in closure
    def make_model(dose_val):
        def model(t, ka, CL, V):
            ke = CL / V
            if np.isclose(ka, ke, rtol=1e-6):
                return (F * dose_val * ka / V) * t * np.exp(-ke * t)
            C = (F * dose_val * ka) / (V * (ka - ke)) * (np.exp(-ke * t) - np.exp(-ka * t))
            return C
        return np.vectorize(model, excluded=['ka', 'CL', 'V'])

    model_s = make_model(dose_s)

    def fit_fn(t, ka, CL, V):
        return model_s(t, ka=ka, CL=CL, V=V)

    try:
        popt_s, _ = curve_fit(fit_fn, t_s, C_s, p0=p0, bounds=bounds, maxfev=10000)
        ka_s, CL_s, V_s = popt_s
        ke_s = CL_s / V_s
        t_half_s = np.log(2) / ke_s
        results.append({
            'Subject': subj_id,
            'Weight_kg': bw_s,
            'Dose_mg_kg': dose_s,
            'ka_h': round(ka_s, 3),
            'CL_L_h_kg': round(CL_s, 4),
            'V_L_kg': round(V_s, 3),
            'ke_h': round(ke_s, 4),
            't_half_h': round(t_half_s, 2)
        })
    except RuntimeError as e:
        print(f'Subject {subj_id}: fit failed — {e}')

params_df = pd.DataFrame(results)
print(params_df.to_string(index=False))

### Visualising parameter variability

Now let us plot the distribution of $CL$ and $V$ across subjects. With only 6 subjects a histogram is sparse, so we use a **strip plot** (individual points) alongside the mean.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

param_info = [
    ('ka_h',       'ka (h⁻¹)',          'Absorption Rate Constant'),
    ('CL_L_h_kg',  'CL (L/h/kg)',        'Clearance'),
    ('V_L_kg',     'V (L/kg)',            'Volume of Distribution'),
]

for ax, (col, ylabel, title) in zip(axes, param_info):
    vals = params_df[col].values
    subj_ids = params_df['Subject'].values

    # Strip plot: jitter x slightly so points don't stack
    np.random.seed(0)
    x_jitter = np.random.uniform(-0.08, 0.08, len(vals))
    ax.scatter(x_jitter, vals, s=80, zorder=3, color='steelblue', edgecolors='white')

    # Annotate each point with subject number
    for xi, yi, sid in zip(x_jitter, vals, subj_ids):
        ax.text(xi + 0.05, yi, str(sid), fontsize=8, va='center')

    # Mean line
    mean_val = vals.mean()
    ax.axhline(mean_val, color='crimson', linestyle='--', linewidth=1.5,
               label=f'Mean = {mean_val:.4f}')

    ax.set_xlim(-0.5, 0.5)
    ax.set_xticks([])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('Population PK: Fitted Parameters Across 6 Subjects', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nSummary statistics:')
print(params_df[['ka_h', 'CL_L_h_kg', 'V_L_kg', 't_half_h']].describe().round(4))

### Interpretation

Even though all subjects received nearly the same dose, their PK parameters differ:

- **$k_a$** shows moderate variability — absorption speed differs between individuals (gut motility, formulation interaction, etc.)
- **$CL$** shows variability that could reflect differences in hepatic enzyme activity (theophylline is mainly metabolised by CYP1A2)
- **$V$** is relatively consistent — volume of distribution is largely determined by tissue binding and physicochemical properties, which vary less between healthy adults

### What a full population model would add

The two-stage approach we used here fits each subject independently and then looks at the spread. A **nonlinear mixed-effects (NLME)** model does this in a single step by assuming each subject's parameters come from a population distribution:

$$\theta_i = \theta_{\text{pop}} \cdot e^{\eta_i}, \qquad \eta_i \sim \mathcal{N}(0, \omega^2)$$

This is more statistically rigorous (especially with sparse data) and allows you to explain part of the variability with **covariates** like body weight, age, or genotype. NLME is the engine behind dose individualisation in clinical pharmacology.

---
# Part 5: PD Modeling — The Emax Model

## From concentration to effect

So far we have modeled how concentration changes over time (PK). Now we ask: **given a concentration, what is the pharmacological effect?**

The most widely used PD model is the **Emax model**, derived from receptor occupancy theory. Assume a drug binds to a receptor reversibly:

$$\text{Drug} + \text{Receptor} \rightleftharpoons \text{Drug-Receptor complex} \rightarrow \text{Effect}$$

At equilibrium, the fraction of receptors occupied follows the **Hill equation**:

$$E(C) = \frac{E_{\max} \cdot C}{EC_{50} + C}$$

| Parameter | Meaning |
|-----------|--------|
| $E_{\max}$ | Maximum possible effect (e.g., 100% inhibition) |
| $EC_{50}$ | Concentration producing **half-maximal** effect — a measure of potency |

**Key property — saturation:** When $C \gg EC_{50}$, the denominator $\approx C$ and $E \approx E_{\max}$. Doubling the dose near saturation gives almost no additional effect. This is why simply "giving more drug" is not always useful.

### Generating synthetic PD data

We will:
1. Use the PK profile of Subject 1 (concentrations predicted by our fitted model) as input concentrations
2. Apply the Emax model with **true** parameters $E_{\max} = 100$, $EC_{50} = 5$ mg/L
3. Add Gaussian noise to simulate measurement error
4. Then **fit** the Emax model back to the noisy data to see how well we recover the parameters

In [ ]:
# --- Generate concentration time course for Subject 1 using fitted PK ---
t_pd = np.linspace(0, 24, 200)

# Rebuild the model for Subject 1 (dose = 4.02 mg/kg, F = 1)
ka_s1, CL_s1, V_s1 = params_df[params_df['Subject'] == 1][['ka_h', 'CL_L_h_kg', 'V_L_kg']].values[0]
ke_s1 = CL_s1 / V_s1
dose_s1_val = 4.02

C_pk = (F * dose_s1_val * ka_s1) / (V_s1 * (ka_s1 - ke_s1)) * \
       (np.exp(-ke_s1 * t_pd) - np.exp(-ka_s1 * t_pd))

# --- True PD parameters ---
Emax_true = 100.0   # %
EC50_true = 5.0     # mg/L

def emax_model(C, Emax, EC50):
    return Emax * C / (EC50 + C)

# --- Add Gaussian noise ---
np.random.seed(42)
noise_std = 3.0
E_noisy = emax_model(C_pk, Emax_true, EC50_true) + np.random.normal(0, noise_std, len(C_pk))
E_noisy = np.clip(E_noisy, 0, 105)   # effects cannot be negative or >105%

print(f'Concentration range: {C_pk.min():.2f} – {C_pk.max():.2f} mg/L')
print(f'Effect range (noisy): {E_noisy.min():.1f} – {E_noisy.max():.1f} %')

### Fitting the Emax model

We fit the Emax model to the concentration-effect pairs (not to time directly). This is a simple 2-parameter nonlinear fit.

In [ ]:
# Fit Emax model: E vs C
popt_pd, _ = curve_fit(emax_model, C_pk, E_noisy, p0=[90, 3], bounds=([0, 0], [200, 50]))
Emax_fit, EC50_fit = popt_pd

print('=== Fitted PD parameters ===')
print(f'  Emax  (true={Emax_true})  fitted={Emax_fit:.2f} %')
print(f'  EC50  (true={EC50_true})  fitted={EC50_fit:.2f} mg/L')

# --- Plot 1: Concentration-Effect curve ---
C_range = np.linspace(0, 12, 300)
E_curve_true  = emax_model(C_range, Emax_true, EC50_true)
E_curve_fit   = emax_model(C_range, Emax_fit,  EC50_fit)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: C–E relationship
axes[0].scatter(C_pk, E_noisy, color='steelblue', alpha=0.5, s=15, label='Simulated data')
axes[0].plot(C_range, E_curve_true, 'k--', linewidth=1.5, label=f'True (EC50={EC50_true})')
axes[0].plot(C_range, E_curve_fit,  'r-',  linewidth=2,   label=f'Fitted (EC50={EC50_fit:.2f})')
axes[0].axvline(EC50_fit, color='red', alpha=0.3, linestyle=':', label=f'EC50 = {EC50_fit:.2f} mg/L')
axes[0].axhline(Emax_fit / 2, color='red', alpha=0.3, linestyle=':')
axes[0].set_xlabel('Concentration (mg/L)')
axes[0].set_ylabel('Effect (%)')
axes[0].set_title('Concentration–Effect Relationship')
axes[0].legend(fontsize=9)
axes[0].set_xlim(left=0)
axes[0].set_ylim(0, 110)

# Right: Effect vs time
E_time_fit = emax_model(C_pk, Emax_fit, EC50_fit)
ax2_twin = axes[1].twinx()
axes[1].plot(t_pd, C_pk, 'b-', linewidth=2, label='Concentration (mg/L)')
ax2_twin.plot(t_pd, E_time_fit, 'r-', linewidth=2, label='Effect (%)')
ax2_twin.scatter(t_pd, E_noisy, color='salmon', alpha=0.4, s=10)
axes[1].set_xlabel('Time (h)')
axes[1].set_ylabel('Concentration (mg/L)', color='blue')
ax2_twin.set_ylabel('Effect (%)', color='red')
axes[1].tick_params(axis='y', labelcolor='blue')
ax2_twin.tick_params(axis='y', labelcolor='red')
axes[1].set_title('Concentration and Effect vs Time')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.tight_layout()
plt.suptitle('PD Modeling: Emax Model', y=1.02, fontsize=13, fontweight='bold')
plt.show()

### Interpretation

**Left panel (C–E curve):** The Emax model describes a **hyperbolic** concentration-effect relationship. The fitted curve closely recovers the true parameters despite the added noise. The EC50 marks the inflection point: at concentrations well below EC50, the effect increases roughly linearly with concentration; at concentrations well above EC50, the effect plateaus near Emax.

**Right panel (Effect vs time):** Effect follows concentration through time — it rises as the drug is absorbed and falls as the drug is eliminated. However, because the PK peak concentration (~10 mg/L) is near the EC50 (~5 mg/L), the effect does not simply mirror the concentration curve: the rising and falling portions are asymmetric due to saturation.

**Saturation and the diminishing returns principle:** If $EC_{50} = 5$ mg/L and a dose produces $C_{\max} = 10$ mg/L (2× EC50), the effect is $10/(5+10) = 67\%$ of $E_{\max}$. To reach 90% of $E_{\max}$, you need $C = 9 \times EC_{50} = 45$ mg/L — a 4.5-fold higher concentration for only a 23-percentage-point gain in effect. This is the fundamental reason why dose-response relationships are not linear.

---
# Part 6: Numerical ODE Solvers

## When analytic solutions are not enough

All the solutions we have used so far were **analytic** — closed-form formulas derived by solving the ODE by hand. This works beautifully for simple linear systems (first-order absorption, first-order elimination).

But real biology is often non-linear:
- **Saturable (Michaelis-Menten) elimination:** enzymes have finite capacity; when the drug concentration exceeds $K_m$, elimination becomes zero-order
- **Auto-induction/inhibition:** the drug changes the expression of its own metabolising enzyme
- **Feedback loops:** the drug effect alters a physiological variable that in turn affects the drug's own disposition

For these cases, we need **numerical ODE solvers**. Python provides `scipy.integrate.solve_ivp` — it can handle any system of ODEs you can write down.

### Step 1: Implement the oral model with solve_ivp

Let us first re-implement the 1-compartment oral model as a system of ODEs and verify it matches the analytic solution exactly. This validates our solver setup before we move to non-linear models.

In [ ]:
# --- ODE system: 1-compartment oral model ---
# State vector: y = [D_gut, C_plasma]
# dD_gut/dt = -ka * D_gut
# dC/dt     = ka * D_gut / V - ke * C

def odes_1cpt_oral(t, y, ka, ke, V):
    D_gut, C = y
    dD_gut = -ka * D_gut
    dC     = ka * D_gut / V - ke * C
    return [dD_gut, dC]

# Parameters (Subject 1 fitted values)
ka_num = ka_s1
ke_num = ke_s1
V_num  = V_s1
dose_num = dose_s1_val   # mg/kg

# Initial conditions: drug all in gut, plasma empty
y0 = [F * dose_num, 0.0]

# Time span and evaluation points
t_span = (0, 24)
t_eval = np.linspace(0, 24, 400)

# Solve numerically
sol = solve_ivp(
    fun=odes_1cpt_oral,
    t_span=t_span,
    y0=y0,
    t_eval=t_eval,
    args=(ka_num, ke_num, V_num),
    method='RK45',
    rtol=1e-8,
    atol=1e-10
)

C_numerical = sol.y[1]  # plasma concentration from ODE solver

# Compute analytic solution for the same time vector
C_analytic = (F * dose_num * ka_num) / (V_num * (ka_num - ke_num)) * \
             (np.exp(-ke_num * t_eval) - np.exp(-ka_num * t_eval))

# --- Plot: overlay numerical vs analytic ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(t_eval, C_analytic,  'b-',  linewidth=2.5, label='Analytic solution', alpha=0.8)
axes[0].plot(t_eval, C_numerical, 'r--', linewidth=1.5, label='solve_ivp (RK45)',  alpha=0.9)
axes[0].scatter(t_obs, C_obs, color='black', zorder=5, s=50, label='Subject 1 data')
axes[0].set_xlabel('Time (h)')
axes[0].set_ylabel('Concentration (mg/L)')
axes[0].set_title('Analytic vs Numerical: 1-Cpt Oral Model')
axes[0].legend()
axes[0].set_ylim(bottom=0)

# Residuals between methods
resid = C_numerical - C_analytic
axes[1].plot(t_eval, resid * 1e6, 'purple', linewidth=1)
axes[1].axhline(0, color='grey', linestyle=':')
axes[1].set_xlabel('Time (h)')
axes[1].set_ylabel('Difference (× 10⁻⁶ mg/L)')
axes[1].set_title('Numerical − Analytic Residual (should be ~zero)')

plt.tight_layout()
plt.show()

print(f'Max absolute difference: {np.max(np.abs(resid)):.2e} mg/L  (effectively machine precision)')

### Interpretation

The numerical solution (dashed red) and the analytic solution (solid blue) are visually indistinguishable. The right panel confirms that the difference is on the order of $10^{-6}$ mg/L or smaller — essentially floating-point machine precision. This gives us confidence that our ODE system is correctly coded and that `solve_ivp` is an accurate solver for this problem.

### Step 2: Non-linear elimination — Michaelis-Menten kinetics

Now let us add **saturable elimination** to the model. Instead of a linear elimination term ($-k_e \cdot C$), we use the Michaelis-Menten expression:

$$\frac{dC}{dt} = \frac{k_a \cdot D_{\text{gut}}}{V} - \frac{V_{\max} \cdot C}{K_m + C}$$

where:
- $V_{\max}$ = maximum elimination rate (mg/L/h)
- $K_m$ = Michaelis constant = concentration at half-maximal elimination rate

When $C \ll K_m$: elimination $\approx (V_{\max}/K_m) \cdot C$ — behaves like first-order (linear)
When $C \gg K_m$: elimination $\approx V_{\max}$ — zero-order (constant rate regardless of concentration)

This is also called **capacity-limited** or **non-linear** PK. Phenytoin (anti-epileptic) is a classic example — its enzymes saturate within the therapeutic range, making dose adjustments dangerous.

**There is no simple analytic solution for this system.** We must use the numerical solver.

In [ ]:
# --- ODE system with Michaelis-Menten elimination ---
def odes_mm(t, y, ka, V, Vmax, Km):
    D_gut, C = y
    dD_gut = -ka * D_gut
    dC     = ka * D_gut / V - Vmax * C / (Km + C)
    return [dD_gut, dC]

# We will compare three scenarios:
# 1. Linear elimination (first-order) — our standard model
# 2. MM with Km >> C_peak (unsaturated — should look like linear)
# 3. MM with Km ~ C_peak (saturated — non-linear terminal phase)

ka_mm   = ka_s1
V_mm    = V_s1
dose_mm = dose_s1_val
y0_mm   = [F * dose_mm, 0.0]
t_eval_mm = np.linspace(0, 36, 600)   # extend to 36h to see the difference

# Scenario 1: linear (as before)
C_linear = (F * dose_mm * ka_mm) / (V_mm * (ka_mm - ke_s1)) * \
           (np.exp(-ke_s1 * t_eval_mm) - np.exp(-ka_mm * t_eval_mm))

# Scenario 2: MM, low saturation (Km = 50 mg/L >> C_peak ~ 10 mg/L)
# Set Vmax so that Vmax/Km ~ ke (same low-concentration behaviour)
Km_low  = 50.0                     # mg/L — well above C_peak
Vmax_low = ke_s1 * Km_low          # L/h/kg * mg/L = mg/h/kg

sol_mm_low = solve_ivp(
    odes_mm, (0, 36), y0_mm, t_eval=t_eval_mm,
    args=(ka_mm, V_mm, Vmax_low, Km_low),
    method='RK45', rtol=1e-9, atol=1e-11
)
C_mm_low = sol_mm_low.y[1]

# Scenario 3: MM, moderate saturation (Km = 5 mg/L ~ C_peak)
Km_sat  = 5.0                      # mg/L — within therapeutic range
Vmax_sat = ke_s1 * Km_sat          # matched at low C for fair comparison

sol_mm_sat = solve_ivp(
    odes_mm, (0, 36), y0_mm, t_eval=t_eval_mm,
    args=(ka_mm, V_mm, Vmax_sat, Km_sat),
    method='RK45', rtol=1e-9, atol=1e-11
)
C_mm_sat = sol_mm_sat.y[1]

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Linear scale
axes[0].plot(t_eval_mm, C_linear,   'b-',  linewidth=2,   label='Linear (1st-order)')
axes[0].plot(t_eval_mm, C_mm_low,   'g--', linewidth=1.5, label=f'MM (Km={Km_low} mg/L, unsaturated)')
axes[0].plot(t_eval_mm, C_mm_sat,   'r-',  linewidth=2,   label=f'MM (Km={Km_sat} mg/L, saturated)')
axes[0].set_xlabel('Time (h)')
axes[0].set_ylabel('Concentration (mg/L)')
axes[0].set_title('Linear vs Michaelis-Menten Elimination')
axes[0].legend(fontsize=9)
axes[0].set_ylim(bottom=0)
axes[0].set_xlim(left=0)

# Semi-log scale — non-linearity shows up as a curved terminal slope
axes[1].semilogy(t_eval_mm, np.clip(C_linear,  1e-3, None), 'b-',  linewidth=2,   label='Linear')
axes[1].semilogy(t_eval_mm, np.clip(C_mm_low,  1e-3, None), 'g--', linewidth=1.5, label=f'MM Km={Km_low} (linear range)')
axes[1].semilogy(t_eval_mm, np.clip(C_mm_sat,  1e-3, None), 'r-',  linewidth=2,   label=f'MM Km={Km_sat} (saturated)')
axes[1].set_xlabel('Time (h)')
axes[1].set_ylabel('log Concentration (mg/L)')
axes[1].set_title('Semi-log: Non-linearity Visible as Curved Slope')
axes[1].legend(fontsize=9)
axes[1].set_xlim(left=0)

plt.tight_layout()
plt.suptitle('Non-Linear PK: Michaelis-Menten Elimination (solve_ivp)', y=1.02, fontsize=13, fontweight='bold')
plt.show()

### Interpretation

**Left panel (linear scale):**
- The **linear model** (blue) and **unsaturated MM** (green, $K_m = 50$ mg/L $\gg C_{\text{peak}}$) are essentially identical — when concentrations are well below $K_m$, MM reduces to first-order.
- The **saturated MM** model (red, $K_m = 5$ mg/L $\approx C_{\text{peak}}$) shows a notably slower early decline — at high concentrations, the elimination machinery is running at near-maximum capacity and cannot clear drug faster regardless of how concentrated it is.

**Right panel (semi-log scale):**
- For linear kinetics, a straight line on a semi-log plot is expected (constant slope = $-k_e$)
- For saturated MM, the terminal slope **curves downward** (becomes steeper) as concentrations fall below $K_m$ and kinetics become first-order again

**Clinical implication:** A drug like phenytoin with $K_m$ within the therapeutic range has:
- **Non-proportional dosing:** a small dose increase can cause a disproportionate concentration increase
- **Variable half-life:** the apparent half-life is longer at high doses
- **Narrow therapeutic index:** small changes in enzyme capacity (e.g., drug interactions, illness) have outsized effects on drug levels

This is exactly the kind of scenario where numerical ODE solvers — not analytic formulas — are essential.

---
# Summary

In this lecture we covered the full arc of PK/PD modeling using Python:

| Part | Key takeaway |
|------|--------------|
| **1** | PK describes what the body does to the drug; PD describes what the drug does to the body. A model links dose → concentration → effect. |
| **2** | Simple compartmental models have analytic solutions. CL and V are the primary parameters; ke and t½ are derived. Oral dosing adds an absorption phase governed by ka. |
| **3** | Real data can be fit with `scipy.optimize.curve_fit`. Theophylline data from 6 subjects shows classic oral PK profiles. |
| **4** | Individual fits reveal inter-subject variability. Population PK (NLME) formalises this into fixed and random effects. |
| **5** | The Emax model links concentration to effect. Saturation means doubling the dose near Emax gives diminishing returns — a fundamental pharmacological principle. |
| **6** | `scipy.integrate.solve_ivp` handles any ODE system numerically. Non-linear (Michaelis-Menten) elimination produces curved semi-log plots and concentration-dependent half-lives — no analytic solution available. |

## What to explore next

- **Two-compartment models** — add a peripheral tissue compartment
- **Effect compartment (hysteresis) models** — when effect lags behind plasma concentration
- **Turnover models (indirect response)** — drug inhibits or stimulates production/loss of a response variable
- **NLME in Python** — packages like `nlmixr2` (R) or experimental Python ports; `PyNONMEM`
- **Bayesian PK** — use `PyMC` or `Stan` for full posterior distributions on parameters

---
*Notebook authored for UConn Systems Biology — Fall 2026*